# RNN Unfolding/Unrolling in Time

## Learning Objectives
---

By the end of this notebook, you should be able to:

- Understand what it means to “unfold” or “unroll” an RNN across time steps.
- Relate the hidden state updates to a sequence of repeated computations.
---

## Some Terminologies
---

Let us conceptualize a simple RNN pipeline by taking the following sentence as RNN input:

<p align="center">
  RNN is a type of Neural Network. RNN can be used to model sequence data.
</p>


<p align="center">
  <img src="https://i.postimg.cc/DwtZsPvL/eg1.png" alt="Sequence Diagram" width="600"/>
</p>



1.  **Corpus (Raw Text Data)**

    This is just your collection of normal English text (all the sentences/paragraphs you want your model to learn from). Here, the input sentence is our corpus.


2.  **Word Tokenization**
    We split the text into tokens (usually words).
    Here:

    ["RNN", "is", "a", "type", "of", "Neural", "Network", "RNN", "can", "be", "used", "to", "model", "sequence", "data"]

    Now we have a list of words. The machine will process them step by step.

3.  **Vocabulary Construction**

    We collect all the unique words from our tokens.
    Vocabulary = the "dictionary" our model knows.
    Example:

    ["RNN", "is", "a", "type", "of", "Neural", "Network", "can", "be", "used", "to", "model", "sequence", "data"]

    Notice how “RNN” appeared twice in the text but in the vocabulary it appears only once.

4.  **Add Special Tokens (For Sequence Control)**

    We add some extra “magic” tokens that help the RNN understand sequences:

    - < SOS > = Start of Sentence

    - < EOS > = End of Sentence

    - < UNK > = Unknown word (if a new word appears later that isn’t in the vocabulary)

    Thus the final tokens are:

  **["< SOS >", "< EOS >", "< UNK >", "RNN", "is", "a", "type", "of", "Neural", "Network", "can", "be", "used", "to", "model", "sequence", "data"]**


## One-Hot Encoding

The next step in the process is to convert the categorical data into binary vector data in a way the RNN can understand. Here, the best way to do it is **one-hot encoding**.

<p align="center">
  <img src="https://i.postimg.cc/MKHVcNpX/eg2.png" alt="One Hot Encoding" width="600"/>
</p>

One-hot encoding creates new columns for each category where **1** means the category is present and **0** means it is not.

In this situation, where the tokens are in the order:

**["< SOS >", "< EOS >", "< UNK >", "RNN", "is", "a", "type", "of", "Neural", "Network", "can", "be", "used", "to", "model", "sequence", "data"]**:

One-hot encoding of **< SOS >** (at time step 1):

$y_1$ = [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],

One-hot encoding of **RNN** (at time step 4):

$y_4$ = [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],

One-hot encoding of **data** :

$y_{17}$ = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],

and so on. All the tokens are converted into vectors in this way.








## Feeding input vectors in RNN:
---

Now that we have our categorical data converted into binary vectors, we can feed them to RNN.

As we know, RNNs are a type of neural network where outputs from previous time steps are taken as inputs for the current time step.


In the last notebook, we looked at the basic RNN structure and the update formula for hidden state:

$$h_t = f(W_{ax}x_t + W_{aa}h_{t-1} + b)$$

where:

$x_t$ : input vector at time

$h_{t-1}$ : hidden state from previous time step

$W_{ax}, W_{aa}$ : weight matrices

$b$ : bias

$f$: activation function (e.g., $tanh$)


Also, the update formula for output is:

$$y_t = g(W_{ya} h_t + b_y)$$

where:
$h_t$ = hidden state at time $t$

$W_{ya}$ = weight matrix connecting hidden state to output

$b_y$ = bias for the output layer

$g$ = activation function (e.g., softmax for classification, identity for regression)



The RNN processes the data sequentially, processing one input at a time step. At each time step $t$, the RNN updates its hidden state using the current input and the previous hidden state:

At time step 1 → it sees "RNN". It is only dependent on "RNN"
$$h_1 = f(W_{ax}x_1 + W_{aa}​h_0 + b)$$
$$y_1 = g(W_{ya} h_1 + b_y)$$

At time step 2 → it sees "is", while remembering "RNN". It is dependent on "RNN is"
$$h_2 = f(W_{ax}x_2 + W_{aa}​h_1 + b)$$
$$y_2 = g(W_{ya} h_2 + b_y)$$

At time step 3 → it sees "a", while remembering "RNN is". It is dependent on "RNN is a"
$$h_3 = f(W_{ax}x_3 + W_{aa}​h_2 + b)$$
$$y_3 = g(W_{ya} h_3 + b_y)$$

…and so on, until the full sequence is processed.

## Understanding RNN Training Step (with Cross-Entropy Loss)
---

<p align="center">
  <img src="https://i.postimg.cc/1t1bXD05/eg3.png" width="600"/>
</p>


###  Model Prediction (Softmax Output)  
---
The RNN processes the input and produces a **probability distribution over the vocabulary** at each time step.  

For this time step, the model predicted:

$$
\hat{y}_t = [0.0054, 0.0054, \mathbf{0.3533}, 0.0163, 0.0217, 0.0272, 0.0326, 0.0380, 0.0435, 0.0489, 0.0543, 0.0598, 0.0652, 0.0707, 0.0761, 0.0815]
$$

Notice that the probability for `"RNN"` is **0.3533**, the highest value, but not equal to 1.0.

---

### 3. Loss Computation (Cross-Entropy)  
We use the **Cross-Entropy Loss** at each time step:

$$
L_t = - y_t \cdot \log(\hat{y}_t)
$$

Since $y_t$ is one-hot, this just picks the probability of the correct word:

$$
L_t = -\log(0.3533) \approx 1.0404
$$

---

### 4. Sequence Loss  
For an entire sequence of $T$ words, the loss is the sum of per-token losses:

$$
L = \sum_{j=1}^T L_j
$$

Here, for just this time step,  
**Loss = 1.0404377246170375**
  

## RNN Unfolding Visualization

As we learned, RNN reuses the **same cell (with the same parameters)** across each time step.  

Let's **unfold** this recurrence over multiple time steps to see how RNNs process sequential data.

<p align="center">
  <img src="https://i.postimg.cc/RZvCccsr/RNNUNROLL.gif" alt="Sequence Diagram" width="1000"/>
</p>

In the visual above, you can see what happens when we **stretch** the single unrolled RNN cell across time steps.

As we know, in the previous example given, the input sentence(corpus) **"RNN is a type of Neural Network. RNN can be used to model sequence data."** is split into individual tokens. Each token  is represented as a one-hot vector, and the input vectors ($x_t$) enter the RNN at each time step.

At time step 1, the RNN receives the first input token ("RNN") in its one-hot encoded vector form. The initial hidden state $h_0$ is usually set to zeros. The RNN computes the first hidden state $h_1$ and the first output $y_1$.

At time step 2, the RNN cell processes the next token ("is"), but notice that it does not start fresh. It takes in the current input vector and the previous hidden state $h_1$ carrying memory of "RNN", and the same cell(with same parameters) is used. The new hidden state $h_2$ now encodes "RNN is".

At time step 3, the token "a" is processed. The hidden state $h_3$ now summarizes "RNN is a". This process repeats for every token until the sequence ends.

The same RNN cell parameters ($W_{ax}$, $W_{aa}$, $W_{ya}$, $b$, $b_y$) are reused at every time step, just the input is changed and hidden state is updated, and the output is different each step.


### References

[1].[A visual guide to recurrent neural networks](https://baivab.medium.com/a-visual-guide-to-recurrent-neural-networks-aee1308f62c2)
